
# MS-ILA Day 05 — TV1-C Full Train Fabric (R0/R1/R2) — Colab T4

Đây là **full training thật**, không phải smoke test.

Pipeline dùng **frozen DINO feature cache**:

`locked cache → Adapter(r,d) → [R2: Context→Local alignment] → Projection → FeatureSelector → MeanFusion → BasicDecoder → BCE+Dice → backward → AdamW.step()`

### Trạng thái hiện tại: chờ `(r,d)` Day-04

Notebook cố ý để `ADAPTER_R = None`, `ADAPTER_D = None` và sẽ **hard-stop trước training**. Không dùng lại `(32,128)` của integration smoke.

Khi có winner Day-04, chỉ điền `(r,d)` và nên trỏ thêm `ADAPTER_SELECTION_REPORT`.

R0/R1/R2 dùng chung category, seed, data split, update budget, optimizer, scheduler, projection, decoder, loss, MeanFusion và Git commit; chỉ representation source thay đổi.

Notebook/runner lưu tối thiểu:
- `training_log.csv` theo từng optimizer step;
- `epoch_log.csv`;
- `best.pt`, `last.pt`;
- `resolved_config.yaml`;
- `run_manifest.json`;
- `preflight_report.json`;
- `sample_anomaly_map.png`.

### Có cần chạy lại TV1-B?

Không cần chạy lại smoke cũ ngay bây giờ vì TV1-B đã PASS. Khi có `(r,d)` cuối cùng, runner này tự chạy **preflight bằng chính `(r,d)` cuối cùng** rồi bỏ model preflight và khởi tạo model training mới từ cùng seed.

Nếu code Day-05, backbone hoặc feature cache thay đổi, hãy chạy lại TV1-B raw-image integration.


In [ ]:

# ============================================================
# 0. COLAB + DRIVE
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, sys, json, yaml, hashlib, subprocess, time, re, shlex
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Chọn Runtime -> Change runtime type -> T4 GPU rồi chạy lại."
    )

DEVICE = "cuda:0"
print("PyTorch:", torch.__version__)
print("GPU    :", torch.cuda.get_device_name(0))


In [ ]:

# ============================================================
# 1. CLONE / UPDATE PRIVATE REPO — feat/w5
# ============================================================
from google.colab import userdata

GITHUB_OWNER = "haduckien-raccoon"
GITHUB_REPO = "msila"
GITHUB_BRANCH = "feat/w5"
PROJECT_ROOT = Path("/content/msila")
REPO_URL = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Tạo Colab Secret GITHUB_TOKEN "
        "(fine-grained PAT có quyền Read private repository)."
    ) from exc

if not GITHUB_TOKEN:
    raise RuntimeError("GITHUB_TOKEN đang rỗng.")

askpass = Path("/tmp/git_askpass_msila.sh")
askpass.write_text(
    '''#!/bin/sh
case "$1" in
  *Username*) echo "$GITHUB_USERNAME" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
  *) echo "" ;;
esac
''',
    encoding="utf-8",
)
askpass.chmod(0o700)

git_env = os.environ.copy()
git_env["GIT_ASKPASS"] = str(askpass)
git_env["GIT_TERMINAL_PROMPT"] = "0"
git_env["GITHUB_USERNAME"] = GITHUB_OWNER
git_env["GITHUB_TOKEN"] = GITHUB_TOKEN

def git(args, cwd=None):
    r = subprocess.run(
        ["git", *args],
        cwd=str(cwd) if cwd else None,
        env=git_env,
        text=True,
        capture_output=True,
    )
    if r.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(args)} failed\n"
            f"stdout:\n{r.stdout}\n"
            f"stderr:\n{r.stderr}"
        )
    return (r.stdout or "").strip()

if not (PROJECT_ROOT / ".git").is_dir():
    if PROJECT_ROOT.exists():
        raise RuntimeError(
            f"{PROJECT_ROOT} tồn tại nhưng không phải Git repo."
        )
    subprocess.run(
        [
            "git", "clone",
            "--branch", GITHUB_BRANCH,
            "--single-branch",
            REPO_URL,
            str(PROJECT_ROOT),
        ],
        env=git_env,
        check=True,
    )
else:
    git(["remote", "set-url", "origin", REPO_URL], cwd=PROJECT_ROOT)
    git(["fetch", "origin", GITHUB_BRANCH], cwd=PROJECT_ROOT)
    current = git(["branch", "--show-current"], cwd=PROJECT_ROOT)
    if current != GITHUB_BRANCH:
        local = git(
            ["branch", "--format=%(refname:short)"],
            cwd=PROJECT_ROOT,
        ).splitlines()
        if GITHUB_BRANCH in local:
            git(["checkout", GITHUB_BRANCH], cwd=PROJECT_ROOT)
        else:
            git(
                ["checkout", "-b", GITHUB_BRANCH, f"origin/{GITHUB_BRANCH}"],
                cwd=PROJECT_ROOT,
            )
    git(["pull", "--ff-only", "origin", GITHUB_BRANCH], cwd=PROJECT_ROOT)

branch = git(["branch", "--show-current"], cwd=PROJECT_ROOT)
commit = git(["rev-parse", "HEAD"], cwd=PROJECT_ROOT)
remote = git(["remote", "get-url", "origin"], cwd=PROJECT_ROOT)

assert branch == GITHUB_BRANCH
assert GITHUB_TOKEN not in remote

print("Branch :", branch)
print("Commit :", commit)
print("Remote :", remote)

askpass.unlink(missing_ok=True)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
# ============================================================
# 2. INSTALL TV1-C RUNNER INTO src/train/screen_representation.py
# ============================================================
# Không sửa model/data/loss hiện có; chỉ thêm runner TV1-C.

RUNNER_SOURCE = '#!/usr/bin/env python3\nfrom __future__ import annotations\n\nimport argparse, csv, hashlib, json, math, os, random, subprocess, time\nfrom pathlib import Path\nfrom typing import Any, Mapping, Sequence\n\nimport numpy as np\nimport torch\nimport yaml\nfrom PIL import Image\nfrom torch import Tensor, nn\n\nfrom src.data.cached_dataset import CachedFeatureDataset, load_training_records, make_cached_dataloader\nfrom src.data.feature_cache import FeatureCacheReader, validate_cache\nfrom src.losses.anomaly_loss import AnomalySegmentationLoss\nfrom src.models.basic_decoder import BasicDecoder\nfrom src.models.cached_training import cached_meta_to_aligner_geometry\nfrom src.models.context_alignment import ContextToLocalAligner\nfrom src.models.feature_projection import SixFeatureProjection\nfrom src.models.feature_selector import FeatureSelector\nfrom src.models.mean_fusion import MeanFusion\nfrom src.models.residual_adapter import ResidualAdapter2d\n\n\nclass FullTrainError(RuntimeError):\n    pass\n\n\ndef _jsonable(x):\n    if isinstance(x, Path):\n        return str(x)\n    if isinstance(x, Mapping):\n        return {str(k): _jsonable(v) for k, v in x.items()}\n    if isinstance(x, (list, tuple)):\n        return [_jsonable(v) for v in x]\n    return x\n\n\ndef canonical_json(x):\n    return json.dumps(_jsonable(x), sort_keys=True, separators=(",", ":"), ensure_ascii=False)\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef sha256_json(x):\n    return hashlib.sha256(canonical_json(x).encode("utf-8")).hexdigest()\n\n\ndef save_json(obj, path: Path):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(_jsonable(obj), indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n    os.replace(tmp, path)\n\n\ndef save_yaml(obj, path: Path):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(yaml.safe_dump(_jsonable(obj), sort_keys=False, allow_unicode=True), encoding="utf-8")\n    os.replace(tmp, path)\n\n\ndef read_yaml(path):\n    obj = yaml.safe_load(Path(path).read_text(encoding="utf-8"))\n    if not isinstance(obj, dict):\n        raise FullTrainError(f"Expected YAML mapping: {path}")\n    return obj\n\n\ndef git_value(*args):\n    try:\n        return subprocess.check_output(["git", *args], text=True, stderr=subprocess.DEVNULL).strip() or None\n    except Exception:\n        return None\n\n\ndef seed_everything(seed: int):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.use_deterministic_algorithms(True, warn_only=False)\n    torch.backends.cudnn.benchmark = False\n\n\ndef record_split(r):\n    value = r.get("split")\n    if value is None and isinstance(r.get("meta"), Mapping):\n        value = r["meta"].get("split")\n    return None if value is None else str(value).strip().lower().replace("-", "_")\n\n\ndef filter_records(records, category: str, role: str):\n    out = [dict(r) for r in records if str(r.get("category", "")).lower() == category.lower()]\n    if not out:\n        raise FullTrainError(f"{role}: no records for category={category}")\n    splits = {record_split(r) for r in out if record_split(r)}\n    forbidden = {"overfit16", "test_public", "test_private", "test_private_mixed", "validation"}\n    if splits & forbidden:\n        raise FullTrainError(f"{role}: forbidden splits: {sorted(splits & forbidden)}")\n    if role == "val" and splits != {"dev_synthetic"}:\n        raise FullTrainError(f"val must be DEV-synthetic, got {sorted(splits)}")\n    if role == "train" and splits and not splits.issubset({"train", "train_core"}):\n        raise FullTrainError(f"train must be TRAIN-core/train, got {sorted(splits)}")\n    return out\n\n\ndef records_fingerprint(records):\n    rows = []\n    for r in records:\n        rows.append({\n            "image_id": str(r.get("image_id")),\n            "category": str(r.get("category")),\n            "mask_path": None if r.get("mask_path") is None else str(r.get("mask_path")),\n            "mask_hw": r.get("mask_hw"),\n            "is_anomaly": r.get("is_anomaly"),\n            "split": record_split(r),\n        })\n    return sha256_json(rows)\n\n\ndef known_positive_count(records):\n    flags = []\n    for r in records:\n        if "is_anomaly" in r:\n            flags.append(bool(r["is_anomaly"]))\n        elif isinstance(r.get("meta"), Mapping) and "is_anomaly" in r["meta"]:\n            flags.append(bool(r["meta"]["is_anomaly"]))\n    return None if not flags else sum(flags)\n\n\ndef verify_cache(cache_dir: Path, train_records, val_records, expected_backbone: str, allow_unverified=False):\n    validate_cache(cache_dir)\n    reader = FeatureCacheReader(cache_dir, mmap=True, shard_cache_size=1)\n    missing = []\n    for r in list(train_records) + list(val_records):\n        key = (str(r["image_id"]), str(r["category"]))\n        if key not in reader:\n            missing.append(key)\n    if missing:\n        raise FullTrainError(f"cache missing {len(missing)} required records; first={missing[:3]}")\n\n    signature = reader.manifest.get("producer_signature")\n    if not isinstance(signature, Mapping):\n        if not allow_unverified:\n            raise FullTrainError("cache manifest lacks producer_signature")\n        return reader, {"verified": False, "reason": "no producer_signature"}\n\n    text = json.dumps(signature, ensure_ascii=False).lower()\n    expected = expected_backbone.lower()\n    if "vitb16" in expected and "vits16" in text:\n        raise FullTrainError("BACKBONE DRIFT: Day-05 expects ViT-B/16 but cache says ViT-S/16")\n    verified = expected in text or ("vitb16" in expected and "vitb16" in text)\n    if not verified and not allow_unverified:\n        raise FullTrainError(f"cannot verify cache backbone={expected_backbone} from producer_signature")\n    return reader, {\n        "verified": bool(verified),\n        "producer_signature": signature,\n        "producer_sha256": reader.manifest.get("producer_sha256"),\n    }\n\n\ndef infer_in_channels(reader, record):\n    sample = reader.get(image_id=str(record["image_id"]), category=str(record["category"]))\n    x = sample["local_b4"]\n    if x.ndim == 4:\n        return int(x.shape[1])\n    if x.ndim == 3:\n        return int(x.shape[0])\n    raise FullTrainError(f"unexpected local_b4 shape: {tuple(x.shape)}")\n\n\ndef move_to_device(x, device):\n    if isinstance(x, Tensor):\n        return x.to(device, non_blocking=True)\n    if isinstance(x, dict):\n        return {k: move_to_device(v, device) for k, v in x.items()}\n    if isinstance(x, list):\n        return [move_to_device(v, device) for v in x]\n    if isinstance(x, tuple):\n        return tuple(move_to_device(v, device) for v in x)\n    return x\n\n\ndef grad_norm(model):\n    total = 0.0\n    seen = False\n    for p in model.parameters():\n        if p.requires_grad and p.grad is not None:\n            seen = True\n            g = p.grad.detach().float()\n            if not bool(torch.isfinite(g).all()):\n                raise FullTrainError("NaN/Inf gradient")\n            total += float(g.pow(2).sum().item())\n    if not seen:\n        raise FullTrainError("no trainable gradient")\n    return math.sqrt(total)\n\n\ndef snapshot_trainable(model):\n    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}\n\n\ndef changed_names(before, model):\n    now = dict(model.named_parameters())\n    return [n for n, old in before.items() if not torch.equal(old, now[n].detach().cpu())]\n\n\nclass Day05RepresentationModel(nn.Module):\n    def __init__(self, *, day05_config, candidate, in_channels, adapter_r, adapter_d):\n        super().__init__()\n        self.candidate = str(candidate).upper()\n        self.selector = FeatureSelector.from_config(day05_config, self.candidate, validate=True)\n        locked = day05_config["locked"]\n        self.blocks = (4, 8, 12)\n        self.active_blocks = (12,) if self.candidate == "R0" else self.blocks\n\n        self.adapters = nn.ModuleDict({\n            f"b{b}": ResidualAdapter2d(\n                in_dim=int(in_channels),\n                bottleneck_dim=int(adapter_r),\n                projection_dim=int(adapter_d),\n                kernel_size=int(locked["adapter"]["kernel_size"]),\n                gamma_init=float(locked["adapter"]["gamma_init"]),\n                bias=True,\n            ) for b in self.blocks\n        })\n        self.aligner = ContextToLocalAligner(check_finite=True, check_bounds=True)\n        self.projection = SixFeatureProjection(\n            in_channels=int(in_channels),\n            fusion_dim=int(locked["projection"]["fusion_dim"]),\n            blocks=self.blocks,\n            share_across_views=bool(locked["projection"]["share_across_views"]),\n            check_finite=True,\n        )\n        self.fusion = MeanFusion(validate=True, return_weights=True)\n        self.decoder = BasicDecoder(in_channels=int(locked["projection"]["fusion_dim"]))\n        self._freeze_inactive()\n\n    def _freeze_inactive(self):\n        active = set(self.active_blocks)\n        for b in self.blocks:\n            if b not in active:\n                self.adapters[f"b{b}"].requires_grad_(False)\n        if not self.projection.share_across_views:\n            raise FullTrainError("Day-05 requires share_projection_across_views=true")\n        for b in self.blocks:\n            if b not in active:\n                self.projection.projectors[f"b{b}"].requires_grad_(False)\n        if any(True for _ in self.fusion.parameters()):\n            raise FullTrainError("MeanFusion must be parameter-free")\n        if any(True for _ in self.aligner.parameters()):\n            raise FullTrainError("Context aligner must be parameter-free")\n\n    def forward(self, batch, retain_source_grads=False):\n        local = {}\n        for b in self.active_blocks:\n            local[f"L{b}"] = self.adapters[f"b{b}"](batch[f"local_b{b}"])\n\n        aligned_context = None\n        if self.candidate == "R2":\n            context = {\n                f"C{b}": self.adapters[f"b{b}"](batch[f"context_b{b}"])\n                for b in self.blocks\n            }\n            geometry = cached_meta_to_aligner_geometry(\n                batch["meta"],\n                device=local["L4"].device,\n                matrix_dtype=torch.float32,\n            )\n            h, w = local["L4"].shape[-2:]\n            aligned_context = self.aligner(context, geometry, target_hw=(int(h), int(w)))\n\n        projected = self.projection.project_sources(\n            local,\n            aligned_context,\n            source_keys=self.selector.source_keys,\n        )\n        if retain_source_grads:\n            for x in projected.values():\n                x.retain_grad()\n\n        selected = self.selector(projected)\n        fused, weights = self.fusion(selected)\n\n        mask = batch["mask"]\n        logits = self.decoder(fused, output_size=(int(mask.shape[-2]), int(mask.shape[-1])))\n        return logits, {\n            "projected": projected,\n            "selected": selected,\n            "fused": fused,\n            "fusion_weights": weights,\n        }\n\n\ndef build_optimizer(model, protocol):\n    cfg = protocol["training"]["optimizer"]\n    if str(cfg["name"]) != "AdamW":\n        raise FullTrainError(f"locked optimizer must be AdamW, got {cfg[\'name\']}")\n    params = [p for p in model.parameters() if p.requires_grad]\n    return torch.optim.AdamW(params, lr=float(cfg["lr"]), **dict(cfg.get("kwargs", {})))\n\n\ndef verify_protocol(protocol):\n    t = protocol["training"]\n    for k in ("epochs", "batch_size", "optimizer"):\n        if t.get(k) is None:\n            raise FullTrainError(f"training.{k} is unresolved/null")\n    if t.get("scheduler") not in (None, False):\n        raise FullTrainError("non-null scheduler not implemented: lock it explicitly before training")\n    if int(t["epochs"]) <= 0 or int(t["batch_size"]) <= 0:\n        raise FullTrainError("epochs/batch_size must be >0")\n    if float(t["optimizer"]["lr"]) <= 0:\n        raise FullTrainError("optimizer lr must be >0")\n\n\ndef make_dataset(cache_dir, records, protocol, mask_root):\n    d = protocol["data"]\n    return CachedFeatureDataset(\n        cache_dir=cache_dir,\n        records=records,\n        expected_producer_signature=None,\n        mask_root=mask_root,\n        mask_threshold=int(d.get("mask_threshold", 0)),\n        mask_hw_source=str(d.get("mask_hw_source", "record")),\n        allow_zero_mask_for_normal=bool(d.get("allow_zero_mask_for_normal", True)),\n        squeeze_cached_batch_dim=bool(d.get("squeeze_cached_batch_dim", True)),\n        feature_dtype=torch.float32,\n        mmap=bool(d.get("mmap", True)),\n        shard_cache_size=int(d.get("shard_cache_size", 2)),\n    )\n\n\ndef make_loader(ds, protocol, shuffle, seed):\n    d = protocol["data"]\n    t = protocol["training"]\n    return make_cached_dataloader(\n        ds,\n        batch_size=int(t["batch_size"]),\n        shuffle=shuffle,\n        num_workers=int(d.get("num_workers", 0)),\n        pin_memory=d.get("pin_memory", True),\n        persistent_workers=d.get("persistent_workers", False),\n        prefetch_factor=int(d.get("prefetch_factor", 2)),\n        drop_last=bool(d.get("drop_last", False)) if shuffle else False,\n        seed=int(seed),\n    )\n\n\ndef make_criterion(day05):\n    loss = day05["locked"]["loss"]\n    if float(loss.get("illumination_weight", 0.0)) != 0.0:\n        raise FullTrainError("illumination loss is forbidden in Day-05 representation ablation")\n    return AnomalySegmentationLoss(\n        bce_weight=float(loss["bce_weight"]),\n        dice_weight=float(loss["dice_weight"]),\n    )\n\n\ndef source_usage(trace):\n    out = {}\n    for key, tensor in trace["projected"].items():\n        g = tensor.grad\n        out[key] = {\n            "grad_present": g is not None,\n            "finite": bool(torch.isfinite(g).all()) if g is not None else False,\n            "grad_l1": float(g.detach().abs().sum().item()) if g is not None else 0.0,\n        }\n    return out\n\n\ndef run_preflight(model, loader, criterion, optimizer, device, steps):\n    if steps < 2:\n        raise FullTrainError("preflight_steps must be >=2 because gamma_init=0")\n    before = snapshot_trainable(model)\n    it = iter(loader)\n    reports, losses = [], []\n    model.train()\n\n    for _ in range(steps):\n        try:\n            batch = next(it)\n        except StopIteration:\n            it = iter(loader)\n            batch = next(it)\n        batch = move_to_device(batch, device)\n        optimizer.zero_grad(set_to_none=True)\n        logits, trace = model(batch, retain_source_grads=True)\n        lo = criterion(logits, batch["mask"])\n        loss = lo["loss"]\n        if not bool(torch.isfinite(loss)):\n            raise FullTrainError("preflight loss NaN/Inf")\n        loss.backward()\n        usage = source_usage(trace)\n        for key, row in usage.items():\n            if not row["grad_present"] or not row["finite"] or row["grad_l1"] <= 0:\n                raise FullTrainError(f"source disconnected in preflight: {key}: {row}")\n        if model.candidate == "R2":\n            expected = {\n                "local_b4", "local_b8", "local_b12",\n                "context_b4", "context_b8", "context_b12",\n            }\n            if set(usage) != expected:\n                raise FullTrainError(f"R2 did not use exactly 6 sources: {sorted(usage)}")\n        gn = grad_norm(model)\n        if not math.isfinite(gn) or gn <= 0:\n            raise FullTrainError(f"invalid grad_norm={gn}")\n        optimizer.step()\n        reports.append(usage)\n        losses.append(float(loss.detach().cpu()))\n\n    changed = changed_names(before, model)\n    for prefix in ("adapters.", "projection.", "decoder."):\n        if not any(n.startswith(prefix) for n in changed):\n            raise FullTrainError(f"preflight: no weight changed under {prefix}")\n    return {\n        "status": "PASS",\n        "candidate": model.candidate,\n        "steps": int(steps),\n        "losses": losses,\n        "source_usage": reports,\n        "changed_parameter_count": len(changed),\n        "changed_parameters": changed,\n    }\n\n\ndef append_csv(path: Path, row):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    exists = path.is_file() and path.stat().st_size > 0\n    with path.open("a", newline="", encoding="utf-8") as f:\n        w = csv.DictWriter(f, fieldnames=list(row.keys()))\n        if not exists:\n            w.writeheader()\n        w.writerow(row)\n\n\ndef save_checkpoint(path, model, optimizer, epoch, global_step, best_val, config_hash):\n    payload = {\n        "schema": "msila.day05.full_train.checkpoint.v1",\n        "model": model.state_dict(),\n        "optimizer": optimizer.state_dict(),\n        "epoch": int(epoch),\n        "global_step": int(global_step),\n        "best_val": best_val,\n        "resolved_config_sha256": config_hash,\n        "rng": {\n            "python": random.getstate(),\n            "numpy": np.random.get_state(),\n            "torch_cpu": torch.get_rng_state(),\n            "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,\n        },\n    }\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    torch.save(payload, tmp)\n    os.replace(tmp, path)\n\n\ndef load_checkpoint(path, model, optimizer, expected_hash, device):\n    p = torch.load(path, map_location=device, weights_only=False)\n    if p.get("resolved_config_sha256") != expected_hash:\n        raise FullTrainError("resume config mismatch")\n    model.load_state_dict(p["model"], strict=True)\n    optimizer.load_state_dict(p["optimizer"])\n    rng = p.get("rng", {})\n    if rng:\n        random.setstate(rng["python"])\n        np.random.set_state(rng["numpy"])\n        torch.set_rng_state(rng["torch_cpu"])\n        if torch.cuda.is_available() and rng.get("torch_cuda") is not None:\n            torch.cuda.set_rng_state_all(rng["torch_cuda"])\n    return int(p["epoch"]), int(p["global_step"]), p.get("best_val")\n\n\ndef validate(model, loader, criterion, device):\n    model.eval()\n    sums = {"total_loss": 0.0, "bce": 0.0, "dice": 0.0}\n    n = 0\n    with torch.no_grad():\n        for batch in loader:\n            batch = move_to_device(batch, device)\n            logits, _ = model(batch)\n            out = criterion(logits, batch["mask"])\n            if not bool(torch.isfinite(out["loss"])):\n                raise FullTrainError("validation loss NaN/Inf")\n            sums["total_loss"] += float(out["loss"].detach().cpu())\n            sums["bce"] += float(out["bce"].detach().cpu())\n            sums["dice"] += float(out["dice"].detach().cpu())\n            n += 1\n    if n == 0:\n        raise FullTrainError("validation loader empty")\n    return {k: v / n for k, v in sums.items()}\n\n\ndef save_preview(model, loader, device, path):\n    model.eval()\n    batch = move_to_device(next(iter(loader)), device)\n    with torch.no_grad():\n        logits, _ = model(batch)\n        score = torch.sigmoid(logits[0, 0]).detach().cpu().numpy()\n    lo, hi = float(score.min()), float(score.max())\n    vis = np.zeros_like(score, dtype=np.uint8) if hi <= lo else ((score - lo) / (hi - lo) * 255).astype(np.uint8)\n    Image.fromarray(vis, mode="L").save(path)\n    np.save(path.with_suffix(".npy"), score.astype(np.float32))\n\n\ndef lock_payload(resolved):\n    keys = [\n        "category", "seed", "adapter", "backbone", "input", "projection",\n        "fusion", "decoder", "loss", "training", "data_fingerprints",\n        "cache_producer_sha256", "git_commit", "runner_sha256",\n    ]\n    return {k: resolved[k] for k in keys}\n\n\ndef enforce_lock(output_root, payload):\n    path = Path(output_root) / "protocol_lock.json"\n    digest = sha256_json(payload)\n    record = {"sha256": digest, "payload": payload}\n    if not path.is_file():\n        save_json(record, path)\n        return digest\n    old = json.loads(path.read_text(encoding="utf-8"))\n    if old.get("sha256") != digest:\n        raise FullTrainError(\n            f"PROTOCOL DRIFT across R0/R1/R2: old={old.get(\'sha256\')} new={digest}"\n        )\n    return digest\n\n\ndef train_candidate(args):\n    day05 = read_yaml(args.day05_config)\n    protocol = read_yaml(args.training_protocol)\n    verify_protocol(protocol)\n\n    candidate = args.candidate.upper()\n    if candidate not in {"R0", "R1", "R2"}:\n        raise FullTrainError("candidate must be R0/R1/R2")\n    if args.r <= 0 or args.d <= 0:\n        raise FullTrainError("r,d must be final locked Day-04 winner >0")\n    if day05["locked"]["backbone"]["name"] != "dinov3_vitb16":\n        raise FullTrainError("Day-05 full runner is locked to dinov3_vitb16")\n    if day05["locked"]["fusion"]["type"] != "mean" or day05["locked"]["fusion"]["trainable"]:\n        raise FullTrainError("Day-05 must use parameter-free MeanFusion")\n    if day05["experiment"]["candidate_ids"] != ["R0", "R1", "R2"]:\n        raise FullTrainError("candidate config drift")\n\n    train_records = filter_records(load_training_records(args.train_records), args.category, "train")\n    val_records = filter_records(load_training_records(args.val_records), args.category, "val")\n\n    tp, vp = known_positive_count(train_records), known_positive_count(val_records)\n    if tp == 0:\n        raise FullTrainError("TRAIN-core explicitly has zero anomaly samples; dense BCE+Dice training is invalid")\n    if vp == 0:\n        raise FullTrainError("DEV-synthetic explicitly has zero anomaly samples")\n\n    reader, provenance = verify_cache(\n        Path(args.cache_dir),\n        train_records,\n        val_records,\n        day05["locked"]["backbone"]["name"],\n        allow_unverified=args.allow_unverified_cache_provenance,\n    )\n    in_channels = infer_in_channels(reader, train_records[0])\n\n    mask_root = None if args.mask_root is None else Path(args.mask_root)\n    train_ds = make_dataset(Path(args.cache_dir), train_records, protocol, mask_root)\n    val_ds = make_dataset(Path(args.cache_dir), val_records, protocol, mask_root)\n    train_loader = make_loader(train_ds, protocol, True, args.seed)\n    val_loader = make_loader(val_ds, protocol, False, args.seed)\n    if len(train_loader) == 0 or len(val_loader) == 0:\n        raise FullTrainError("empty train/val loader")\n\n    device = torch.device(args.device)\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise FullTrainError("CUDA requested but unavailable")\n\n    runner_path = Path(__file__).resolve()\n    resolved = {\n        "schema": "msila.day05.full_train.resolved.v1",\n        "candidate": candidate,\n        "representation": day05["representations"][candidate],\n        "category": args.category,\n        "seed": int(args.seed),\n        "adapter": {\n            "r": int(args.r),\n            "d": int(args.d),\n            "kernel_size": int(day05["locked"]["adapter"]["kernel_size"]),\n            "gamma_init": float(day05["locked"]["adapter"]["gamma_init"]),\n            "selection_report": args.adapter_selection_report,\n            "selection_report_sha256": (\n                sha256_file(args.adapter_selection_report)\n                if args.adapter_selection_report else None\n            ),\n        },\n        "backbone": day05["locked"]["backbone"],\n        "input": day05["locked"]["input"],\n        "projection": day05["locked"]["projection"],\n        "fusion": day05["locked"]["fusion"],\n        "decoder": day05["locked"]["decoder"],\n        "loss": day05["locked"]["loss"],\n        "training": {\n            "epochs": int(protocol["training"]["epochs"]),\n            "batch_size": int(protocol["training"]["batch_size"]),\n            "optimizer": protocol["training"]["optimizer"],\n            "scheduler": protocol["training"].get("scheduler"),\n            "gradient_clip_norm": protocol["training"].get("gradient_clip_norm"),\n            "updates_per_epoch": len(train_loader),\n            "total_update_budget": int(protocol["training"]["epochs"]) * len(train_loader),\n        },\n        "data_fingerprints": {\n            "train_source_sha256": sha256_file(args.train_records),\n            "val_source_sha256": sha256_file(args.val_records),\n            "train_filtered_sha256": records_fingerprint(train_records),\n            "val_filtered_sha256": records_fingerprint(val_records),\n        },\n        "cache_producer_sha256": reader.manifest.get("producer_sha256"),\n        "cache": {\n            "dir": str(Path(args.cache_dir).resolve()),\n            "manifest_sha256": sha256_file(Path(args.cache_dir) / "manifest.json"),\n            "provenance_check": provenance,\n            "in_channels": in_channels,\n        },\n        "checkpoint_rule": {"monitor": "val_total_loss", "mode": "min"},\n        "git_branch": git_value("branch", "--show-current"),\n        "git_commit": git_value("rev-parse", "HEAD"),\n        "runner_sha256": sha256_file(runner_path),\n        "torch_version": torch.__version__,\n        "cuda_version": torch.version.cuda,\n    }\n\n    output_root = Path(args.output_root)\n    lock_sha = enforce_lock(output_root, lock_payload(resolved))\n    resolved["protocol_lock_sha256"] = lock_sha\n\n    run_dir = output_root / f"seed_{args.seed}" / args.category / candidate\n    run_dir.mkdir(parents=True, exist_ok=True)\n    resolved_path = run_dir / "resolved_config.yaml"\n    resolved_hash = sha256_json(resolved)\n    if resolved_path.is_file():\n        old = read_yaml(resolved_path)\n        if sha256_json(old) != resolved_hash:\n            raise FullTrainError(f"resolved config drift in {run_dir}")\n    else:\n        save_yaml(resolved, resolved_path)\n\n    manifest_path = run_dir / "run_manifest.json"\n    if manifest_path.is_file():\n        old = json.loads(manifest_path.read_text(encoding="utf-8"))\n        if old.get("status") == "COMPLETE":\n            print(f"[SKIP] already COMPLETE: {run_dir}")\n            return run_dir\n\n    manifest = {\n        "schema": "msila.day05.full_train.manifest.v1",\n        "status": "RUNNING",\n        "candidate": candidate,\n        "category": args.category,\n        "seed": int(args.seed),\n        "adapter": {"r": int(args.r), "d": int(args.d)},\n        "sources": list(day05["representations"][candidate]["sources"]),\n        "git_branch": resolved["git_branch"],\n        "git_commit": resolved["git_commit"],\n        "resolved_config_sha256": resolved_hash,\n        "protocol_lock_sha256": lock_sha,\n        "started_at_unix": time.time(),\n    }\n    save_json(manifest, manifest_path)\n\n    criterion = make_criterion(day05).to(device)\n\n    # Disposable exact-final-r,d preflight.\n    seed_everything(args.seed)\n    pf_model = Day05RepresentationModel(\n        day05_config=day05,\n        candidate=candidate,\n        in_channels=in_channels,\n        adapter_r=args.r,\n        adapter_d=args.d,\n    ).to(device)\n    pf_opt = build_optimizer(pf_model, protocol)\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats(device)\n    pf = run_preflight(\n        pf_model, train_loader, criterion, pf_opt, device, args.preflight_steps\n    )\n    pf["peak_vram_mb"] = (\n        float(torch.cuda.max_memory_allocated(device) / (1024**2))\n        if device.type == "cuda" else 0.0\n    )\n    save_json(pf, run_dir / "preflight_report.json")\n    del pf_model, pf_opt\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n\n    # Fresh actual model.\n    seed_everything(args.seed)\n    model = Day05RepresentationModel(\n        day05_config=day05,\n        candidate=candidate,\n        in_channels=in_channels,\n        adapter_r=args.r,\n        adapter_d=args.d,\n    ).to(device)\n    optimizer = build_optimizer(model, protocol)\n\n    train_log = run_dir / "training_log.csv"\n    epoch_log = run_dir / "epoch_log.csv"\n    last_path = run_dir / "last.pt"\n    best_path = run_dir / "best.pt"\n\n    start_epoch, global_step, best_val = 1, 0, None\n    if args.resume and last_path.is_file():\n        done_epoch, global_step, best_val = load_checkpoint(\n            last_path, model, optimizer, resolved_hash, device\n        )\n        start_epoch = done_epoch + 1\n        print(f"[RESUME] {candidate}: epoch={done_epoch}, step={global_step}, best={best_val}")\n\n    epochs = int(protocol["training"]["epochs"])\n    grad_clip = protocol["training"].get("gradient_clip_norm")\n    t0 = time.perf_counter()\n    if device.type == "cuda":\n        torch.cuda.reset_peak_memory_stats(device)\n\n    for epoch in range(start_epoch, epochs + 1):\n        model.train()\n        sums = {"total_loss": 0.0, "bce": 0.0, "dice": 0.0, "grad_norm": 0.0}\n        n = 0\n        e0 = time.perf_counter()\n\n        for step_in_epoch, batch in enumerate(train_loader, start=1):\n            batch = move_to_device(batch, device)\n            optimizer.zero_grad(set_to_none=True)\n            logits, _ = model(batch)\n            out = criterion(logits, batch["mask"])\n            loss = out["loss"]\n            if not bool(torch.isfinite(loss)):\n                raise FullTrainError(f"NaN/Inf loss at epoch={epoch}, step={step_in_epoch}")\n            loss.backward()\n            gn = grad_norm(model)\n            if grad_clip is not None:\n                torch.nn.utils.clip_grad_norm_(\n                    [p for p in model.parameters() if p.requires_grad],\n                    max_norm=float(grad_clip),\n                )\n            optimizer.step()\n\n            global_step += 1\n            n += 1\n            elapsed = time.perf_counter() - t0\n            peak = (\n                float(torch.cuda.max_memory_allocated(device) / (1024**2))\n                if device.type == "cuda" else 0.0\n            )\n            row = {\n                "candidate": candidate,\n                "category": args.category,\n                "epoch": epoch,\n                "step_in_epoch": step_in_epoch,\n                "global_step": global_step,\n                "total_loss": float(loss.detach().cpu()),\n                "bce": float(out["bce"].detach().cpu()),\n                "dice": float(out["dice"].detach().cpu()),\n                "lr": float(optimizer.param_groups[0]["lr"]),\n                "grad_norm": float(gn),\n                "elapsed_sec": float(elapsed),\n                "peak_vram_mb": peak,\n            }\n            append_csv(train_log, row)\n            sums["total_loss"] += row["total_loss"]\n            sums["bce"] += row["bce"]\n            sums["dice"] += row["dice"]\n            sums["grad_norm"] += row["grad_norm"]\n\n        if n == 0:\n            raise FullTrainError("no training batches")\n\n        val = validate(model, val_loader, criterion, device)\n        train_mean = {k: v / n for k, v in sums.items()}\n        epoch_row = {\n            "candidate": candidate,\n            "category": args.category,\n            "epoch": epoch,\n            "global_step": global_step,\n            "train_total_loss": train_mean["total_loss"],\n            "train_bce": train_mean["bce"],\n            "train_dice": train_mean["dice"],\n            "train_grad_norm": train_mean["grad_norm"],\n            "val_total_loss": val["total_loss"],\n            "val_bce": val["bce"],\n            "val_dice": val["dice"],\n            "lr": float(optimizer.param_groups[0]["lr"]),\n            "epoch_elapsed_sec": float(time.perf_counter() - e0),\n            "elapsed_sec": float(time.perf_counter() - t0),\n            "peak_vram_mb": (\n                float(torch.cuda.max_memory_allocated(device) / (1024**2))\n                if device.type == "cuda" else 0.0\n            ),\n        }\n        append_csv(epoch_log, epoch_row)\n\n        if best_val is None or val["total_loss"] < best_val:\n            best_val = float(val["total_loss"])\n            save_checkpoint(best_path, model, optimizer, epoch, global_step, best_val, resolved_hash)\n        save_checkpoint(last_path, model, optimizer, epoch, global_step, best_val, resolved_hash)\n\n        manifest.update({\n            "status": "RUNNING",\n            "completed_epochs": epoch,\n            "global_step": global_step,\n            "best_val_total_loss": best_val,\n            "last_checkpoint": str(last_path),\n            "best_checkpoint": str(best_path),\n            "elapsed_sec": float(time.perf_counter() - t0),\n            "peak_vram_mb": epoch_row["peak_vram_mb"],\n        })\n        save_json(manifest, manifest_path)\n\n        print(\n            f"[{candidate}] epoch {epoch:03d}/{epochs} | "\n            f"train={train_mean[\'total_loss\']:.6f} | "\n            f"val={val[\'total_loss\']:.6f} | best={best_val:.6f} | "\n            f"step={global_step} | VRAM={epoch_row[\'peak_vram_mb\']:.1f}MB"\n        )\n\n    if not best_path.is_file():\n        raise FullTrainError("best.pt missing")\n    best = torch.load(best_path, map_location=device, weights_only=False)\n    model.load_state_dict(best["model"], strict=True)\n    save_preview(model, val_loader, device, run_dir / "sample_anomaly_map.png")\n\n    manifest.update({\n        "status": "COMPLETE",\n        "completed_epochs": epochs,\n        "global_step": global_step,\n        "best_val_total_loss": best_val,\n        "finished_at_unix": time.time(),\n        "total_elapsed_sec": float(time.perf_counter() - t0),\n        "peak_vram_mb": (\n            float(torch.cuda.max_memory_allocated(device) / (1024**2))\n            if device.type == "cuda" else 0.0\n        ),\n        "artifacts": {\n            "training_log": str(train_log),\n            "epoch_log": str(epoch_log),\n            "best_checkpoint": str(best_path),\n            "last_checkpoint": str(last_path),\n            "resolved_config": str(resolved_path),\n            "preflight_report": str(run_dir / "preflight_report.json"),\n            "sample_anomaly_map": str(run_dir / "sample_anomaly_map.png"),\n        },\n    })\n    save_json(manifest, manifest_path)\n    print(f"[PASS] COMPLETE: {run_dir}")\n    return run_dir\n\n\ndef parse_args(argv: Sequence[str] | None = None):\n    p = argparse.ArgumentParser()\n    p.add_argument("--candidate", required=True, choices=["R0", "R1", "R2"])\n    p.add_argument("--category", default="fabric")\n    p.add_argument("--r", required=True, type=int)\n    p.add_argument("--d", required=True, type=int)\n    p.add_argument("--cache-dir", required=True)\n    p.add_argument("--train-records", required=True)\n    p.add_argument("--val-records", required=True)\n    p.add_argument("--mask-root", default=None)\n    p.add_argument("--output-root", required=True)\n    p.add_argument("--day05-config", default="configs/day05_representation.yaml")\n    p.add_argument("--training-protocol", default="configs/day04_train_protocol.yaml")\n    p.add_argument("--adapter-selection-report", default=None)\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--device", default="cuda:0")\n    p.add_argument("--preflight-steps", type=int, default=3)\n    p.add_argument("--resume", action="store_true")\n    p.add_argument("--allow-unverified-cache-provenance", action="store_true")\n    return p.parse_args(argv)\n\n\ndef main(argv=None):\n    args = parse_args(argv)\n    train_candidate(args)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'

RUNNER_PATH = PROJECT_ROOT / "src" / "train" / "screen_representation.py"
RUNNER_PATH.write_text(RUNNER_SOURCE, encoding="utf-8")
subprocess.run([sys.executable, "-m", "py_compile", str(RUNNER_PATH)], check=True)
print("Runner:", RUNNER_PATH)
print("Runner syntax: PASS")


In [ ]:

# ============================================================
# 3. LOAD LOCKED CONFIGS + REGRESSION TEST
# ============================================================
DEFAULT_PATH = PROJECT_ROOT / "configs" / "default.yaml"
DAY05_PATH = PROJECT_ROOT / "configs" / "day05_representation.yaml"
DAY04_PROTOCOL_PATH = PROJECT_ROOT / "configs" / "day04_train_protocol.yaml"

DEFAULT = yaml.safe_load(DEFAULT_PATH.read_text(encoding="utf-8"))
DAY05 = yaml.safe_load(DAY05_PATH.read_text(encoding="utf-8"))
DAY04_PROTOCOL = yaml.safe_load(
    DAY04_PROTOCOL_PATH.read_text(encoding="utf-8")
)

assert DEFAULT["backbone"]["name"] == "dinov3_vitb16"
assert DEFAULT["backbone"]["frozen"] is True
assert tuple(DEFAULT["backbone"]["feature_blocks"]) == (4, 8, 12)

assert DAY05["experiment"]["candidate_ids"] == ["R0", "R1", "R2"]
assert DAY05["locked"]["fusion"]["type"] == "mean"
assert DAY05["locked"]["fusion"]["trainable"] is False
assert DAY05["locked"]["backbone"]["name"] == "dinov3_vitb16"

print("Config locks: PASS")

r = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_day05.py"],
    cwd=PROJECT_ROOT,
    text=True,
)
if r.returncode != 0:
    raise RuntimeError(
        "tests/test_day05.py FAIL — không được full train."
    )

print("Day-05 regression tests: PASS")


In [ ]:

# ============================================================
# 4. FINAL TRAIN CONFIG — CHỪA TRỐNG r,d CHO TỚI DAY-04 WINNER
# ============================================================
CATEGORY = "fabric"
SEED = int(DAY05["locked"]["training"]["seed"])
CANDIDATES = ("R0", "R1", "R2")

# CHƯA CÓ WINNER -> GIỮ None.
# Khi Day-04 selection xong, chỉ sửa 2 dòng này.
ADAPTER_R = None
ADAPTER_D = None

# Nên trỏ tới selection report sau khi winner được khóa.
ADAPTER_SELECTION_REPORT = None

# Có thể để None để auto-discover.
# Nếu auto-discovery mơ hồ, điền path thật trên Drive.
MANUAL_CACHE_DIR = None
MANUAL_TRAIN_RECORDS = None
MANUAL_VAL_RECORDS = None
MANUAL_MASK_ROOT = None

OUTPUT_ROOT = (
    Path(DEFAULT["data"]["mvtec_ad2"]).parent
    / "outputs"
    / "day05"
    / "full_train"
)

PRECHECK_STEPS = 3
RESUME = True

print("CATEGORY   :", CATEGORY)
print("SEED       :", SEED)
print("ADAPTER_R  :", ADAPTER_R)
print("ADAPTER_D  :", ADAPTER_D)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

if ADAPTER_R is None or ADAPTER_D is None:
    print(
        "\n[WAITING] Chưa train — đây là chủ đích. "
        "Chờ Day-04 chốt winner rồi điền r,d."
    )


In [ ]:
# ============================================================
# 4A. TV1-D — LOCK DAY-04 CHECKPOINT RULE BEFORE TRAINING
#     ĐẶT NGAY SAU CELL 4, TRƯỚC CELL 5
# ============================================================
from src.train.screen_adapter import is_better as day04_is_better

def _sha256_path(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


# ------------------------------------------------------------
# 1. LẤY RULE ĐÚNG TỪ DAY-04 PROTOCOL
# ------------------------------------------------------------
checkpoint_cfg = DAY04_PROTOCOL.get("checkpoint")

if not isinstance(checkpoint_cfg, dict):
    raise RuntimeError(
        "Day-04 protocol không có mapping 'checkpoint'. "
        "Không được tự phát minh checkpoint rule cho Day-05."
    )

TV1D_DAY04_MONITOR = str(
    checkpoint_cfg.get("monitor", "")
).strip()

TV1D_MODE = str(
    checkpoint_cfg.get("mode", "")
).strip().lower()

if not TV1D_DAY04_MONITOR:
    raise RuntimeError(
        "Day-04 checkpoint.monitor đang rỗng."
    )

if TV1D_MODE not in {"min", "max"}:
    raise RuntimeError(
        f"Day-04 checkpoint.mode phải là min/max, "
        f"got={TV1D_MODE!r}"
    )


# ------------------------------------------------------------
# 2. MAP TÊN METRIC DAY-04 -> DAY-05
# ------------------------------------------------------------
# Day-04:
#     hooks.step() trả "loss"
#     screen_adapter prefix validation -> "val_loss"
#
# Day-05 TV1-C:
#     cùng total BCE+Dice loss
#     nhưng epoch_log đặt tên rõ hơn -> "val_total_loss"
#
# Đây chỉ là đổi TÊN CỘT, KHÔNG đổi đại lượng chọn checkpoint.
_MONITOR_ALIAS = {
    "val_loss": "val_total_loss",
    "train_loss": "train_total_loss",
}

TV1D_LOG_MONITOR = _MONITOR_ALIAS.get(
    TV1D_DAY04_MONITOR,
    TV1D_DAY04_MONITOR,
)


# ------------------------------------------------------------
# 3. GATE: RUNNER TV1-C HIỆN TẠI PHẢI DÙNG CÙNG RULE
# ------------------------------------------------------------
# Runner hiện tại của notebook chọn:
#     val_total_loss -> min
#
# Nếu Day-04 protocol thay đổi sau này,
# cell này phải FAIL trước khi train.
_RUNNER_MONITOR = "val_total_loss"
_RUNNER_MODE = "min"

if (
    TV1D_LOG_MONITOR != _RUNNER_MONITOR
    or TV1D_MODE != _RUNNER_MODE
):
    raise RuntimeError(
        "TV1-D RULE DRIFT: checkpoint rule Ngày 4 "
        "không còn tương thích với runner TV1-C hiện tại.\n"
        f"Day-04 raw rule : "
        f"monitor={TV1D_DAY04_MONITOR!r}, "
        f"mode={TV1D_MODE!r}\n"
        f"Resolved Day-05: "
        f"monitor={TV1D_LOG_MONITOR!r}, "
        f"mode={TV1D_MODE!r}\n"
        f"Runner hiện tại: "
        f"monitor={_RUNNER_MONITOR!r}, "
        f"mode={_RUNNER_MODE!r}\n"
        "Phải sửa runner trước khi full train; "
        "không được tự đổi checkpoint rule."
    )


# ------------------------------------------------------------
# 4. FINGERPRINT RULE + CHECKPOINT SELECTOR DAY-04
# ------------------------------------------------------------
selector_source = (
    PROJECT_ROOT
    / "src"
    / "train"
    / "screen_adapter.py"
)

DAY04_PROTOCOL_SHA256 = _sha256_path(
    DAY04_PROTOCOL_PATH
)

DAY04_SELECTOR_SHA256 = _sha256_path(
    selector_source
)


# ------------------------------------------------------------
# 5. LOCK ARTIFACT TRƯỚC KHI TRAIN
# ------------------------------------------------------------
TV1D_RULE_LOCK = {
    "schema": "msila.day05.checkpoint_rule_lock.v1",

    "status": "LOCKED_BEFORE_TV1C_RESULTS",

    "rule_source": str(DAY04_PROTOCOL_PATH),
    "rule_source_sha256": DAY04_PROTOCOL_SHA256,

    "selector_source":
        "src.train.screen_adapter.is_better",

    "selector_file": str(selector_source),
    "selector_file_sha256": DAY04_SELECTOR_SHA256,

    "dev_split": "dev_synthetic",

    "day04_monitor": TV1D_DAY04_MONITOR,
    "day05_log_monitor": TV1D_LOG_MONITOR,

    "mode": TV1D_MODE,

    # Day-04 is_better:
    # min: value < best
    # max: value > best
    "comparison": "strict_improvement",

    # Nếu bằng nhau, checkpoint cũ không bị thay.
    "tie_policy": "keep_earliest_epoch",

    "candidate_scope": list(CANDIDATES),
    "category": CATEGORY,
    "seed": int(SEED),
}


# Hash KHÔNG bao gồm timestamp.
rule_payload_json = json.dumps(
    TV1D_RULE_LOCK,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=False,
).encode("utf-8")

TV1D_RULE_SHA256 = hashlib.sha256(
    rule_payload_json
).hexdigest()

TV1D_RULE_LOCK["rule_lock_sha256"] = (
    TV1D_RULE_SHA256
)

TV1D_RULE_LOCK["locked_at_unix"] = time.time()


TV1D_RULE_LOCK_PATH = (
    OUTPUT_ROOT
    / "checkpoint_rule_lock.json"
)

TV1D_RULE_LOCK_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 6. KHÔNG CHO RULE BỊ ĐỔI GIỮA R0/R1/R2
# ------------------------------------------------------------
if TV1D_RULE_LOCK_PATH.is_file():

    old = json.loads(
        TV1D_RULE_LOCK_PATH.read_text(
            encoding="utf-8"
        )
    )

    # Timestamp không thuộc identity của rule.
    old_identity = dict(old)
    old_identity.pop(
        "locked_at_unix",
        None,
    )
    old_hash = old_identity.pop(
        "rule_lock_sha256",
        None,
    )

    current_identity = dict(
        TV1D_RULE_LOCK
    )
    current_identity.pop(
        "locked_at_unix",
        None,
    )
    current_hash = current_identity.pop(
        "rule_lock_sha256",
        None,
    )

    if (
        old_hash != current_hash
        or old_identity != current_identity
    ):
        raise RuntimeError(
            "Checkpoint rule lock đã tồn tại "
            "nhưng nội dung khác. "
            "Không được trộn R0/R1/R2 "
            "với checkpoint rule khác nhau."
        )

    # Giữ timestamp lock đầu tiên.
    TV1D_RULE_LOCK = old
    TV1D_RULE_SHA256 = str(
        old["rule_lock_sha256"]
    )

else:

    TV1D_RULE_LOCK_PATH.write_text(
        json.dumps(
            TV1D_RULE_LOCK,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


print("=" * 90)
print(
    "TV1-D CHECKPOINT RULE LOCK: PASS"
)
print("=" * 90)

print(
    "Rule source       :",
    DAY04_PROTOCOL_PATH,
)

print(
    "Day-04 monitor    :",
    TV1D_DAY04_MONITOR,
)

print(
    "Day-05 log column :",
    TV1D_LOG_MONITOR,
)

print(
    "Mode              :",
    TV1D_MODE,
)

print(
    "Comparator        :",
    "src.train.screen_adapter.is_better",
)

print(
    "Tie policy        :",
    "keep earliest epoch",
)

print(
    "Rule SHA256       :",
    TV1D_RULE_SHA256,
)

print(
    "Lock artifact     :",
    TV1D_RULE_LOCK_PATH,
)

In [ ]:

# ============================================================
# 5. DISCOVER TRAIN-CORE / DEV-SYNTHETIC / FEATURE CACHE
# ============================================================
from collections import Counter
from src.data.feature_cache import FeatureCacheReader
from src.data.cached_dataset import load_training_records

TEAM_ROOT = Path(DEFAULT["data"]["mvtec_ad2"]).parent
SEARCH_ROOTS = [
    TEAM_ROOT / "msila_day04",
    TEAM_ROOT / "msila_day03",
    TEAM_ROOT / "outputs",
]
SEARCH_ROOTS = [p for p in SEARCH_ROOTS if p.is_dir()]

def record_split(r):
    if not isinstance(r, dict):
        return None
    value = r.get("split")
    if value is None and isinstance(r.get("meta"), dict):
        value = r["meta"].get("split")
    return (
        None
        if value is None
        else str(value).lower().replace("-", "_")
    )

def describe_records(path):
    try:
        rows = load_training_records(path)
    except Exception:
        return None

    rows = [
        r for r in rows
        if str(r.get("category", "")).lower() == CATEGORY
    ]
    if not rows:
        return None

    splits = Counter(
        record_split(r)
        for r in rows
        if record_split(r)
    )
    lower = str(path).lower()
    overfit = (
        "overfit16" in lower
        or set(splits) == {"overfit16"}
    )

    train_score = 0
    if "train_core" in lower or "train-core" in lower:
        train_score += 100
    if set(splits).intersection({"train_core", "train"}):
        train_score += 50
    if "train" in path.name.lower():
        train_score += 10
    if overfit:
        train_score = -1000

    val_score = 0
    if "dev_synthetic" in lower or "dev-synthetic" in lower:
        val_score += 100
    if set(splits) == {"dev_synthetic"}:
        val_score += 100
    if "dev" in path.name.lower() or "synthetic" in path.name.lower():
        val_score += 10
    if overfit:
        val_score = -1000

    return {
        "path": path,
        "n": len(rows),
        "splits": dict(splits),
        "train_score": train_score,
        "val_score": val_score,
    }

record_candidates = []
cache_candidates = []

for root in SEARCH_ROOTS:
    for p in root.rglob("*.json"):
        d = describe_records(p)
        if d:
            record_candidates.append(d)
    for p in root.rglob("*.jsonl"):
        d = describe_records(p)
        if d:
            record_candidates.append(d)

    for manifest in root.rglob("manifest.json"):
        try:
            obj = json.loads(
                manifest.read_text(encoding="utf-8")
            )
        except Exception:
            continue

        if obj.get("schema_name") == "msila.feature_cache":
            cache_candidates.append(manifest.parent)

print("Record candidates:", len(record_candidates))
print("Cache candidates :", len(cache_candidates))

def choose_records(manual, role):
    if manual is not None:
        p = Path(manual)
        if not p.is_file():
            raise FileNotFoundError(p)
        return p

    score_key = (
        "train_score"
        if role == "train"
        else "val_score"
    )
    ranked = sorted(
        [x for x in record_candidates if x[score_key] > 0],
        key=lambda x: (x[score_key], x["n"]),
        reverse=True,
    )

    if not ranked:
        raise RuntimeError(
            f"Không auto-discover được {role} records. "
            f"Hãy điền MANUAL_{role.upper()}_RECORDS."
        )

    print(f"\nTop {role} candidates:")
    for x in ranked[:5]:
        print(
            " ",
            x["path"],
            "| splits=", x["splits"],
            "| n=", x["n"],
            "| score=", x[score_key],
        )

    if (
        len(ranked) > 1
        and ranked[0][score_key] == ranked[1][score_key]
        and ranked[0]["n"] == ranked[1]["n"]
    ):
        raise RuntimeError(
            f"Auto-discovery {role} mơ hồ. "
            f"Điền MANUAL_{role.upper()}_RECORDS."
        )

    return ranked[0]["path"]

TRAIN_RECORDS = choose_records(
    MANUAL_TRAIN_RECORDS,
    "train",
)
VAL_RECORDS = choose_records(
    MANUAL_VAL_RECORDS,
    "val",
)

train_rows = [
    r for r in load_training_records(TRAIN_RECORDS)
    if str(r["category"]).lower() == CATEGORY
]
val_rows = [
    r for r in load_training_records(VAL_RECORDS)
    if str(r["category"]).lower() == CATEGORY
]

if {
    record_split(r)
    for r in val_rows
    if record_split(r)
} != {"dev_synthetic"}:
    raise RuntimeError(
        "VAL_RECORDS phải khóa split='dev_synthetic'."
    )

if "overfit16" in {
    record_split(r)
    for r in train_rows + val_rows
    if record_split(r)
}:
    raise RuntimeError(
        "Overfit-16 chỉ là QA, không dùng TV1-C."
    )

required = [
    (str(r["image_id"]), str(r["category"]))
    for r in train_rows + val_rows
]

candidate_caches = (
    [Path(MANUAL_CACHE_DIR)]
    if MANUAL_CACHE_DIR is not None
    else list(dict.fromkeys(cache_candidates))
)

matching = []
for cache in candidate_caches:
    try:
        reader = FeatureCacheReader(
            cache,
            mmap=True,
            shard_cache_size=0,
        )
        missing = [
            (image_id, category)
            for image_id, category in required
            if (image_id, category) not in reader
        ]
        if not missing:
            matching.append(cache)
    except Exception:
        pass

if len(matching) != 1:
    print("Matching caches:", matching)
    raise RuntimeError(
        "Cần đúng 1 feature cache chứa toàn bộ "
        "TRAIN-core + DEV-synthetic. "
        "Nếu mơ hồ, điền MANUAL_CACHE_DIR."
    )

CACHE_DIR = matching[0]

relative_masks = [
    Path(str(r["mask_path"]))
    for r in train_rows + val_rows
    if r.get("mask_path") not in (None, "")
    and not Path(str(r["mask_path"])).is_absolute()
]

if MANUAL_MASK_ROOT is not None:
    MASK_ROOT = Path(MANUAL_MASK_ROOT)
elif not relative_masks:
    MASK_ROOT = None
else:
    roots = [
        TRAIN_RECORDS.parent,
        VAL_RECORDS.parent,
        TEAM_ROOT / "msila_day04",
        TEAM_ROOT / "msila_day03",
    ]
    valid = [
        root
        for root in roots
        if root.is_dir()
        and all((root / p).is_file() for p in relative_masks)
    ]
    if len(valid) != 1:
        print("Possible mask roots:", valid)
        raise RuntimeError(
            "Không resolve duy nhất MASK_ROOT; "
            "điền MANUAL_MASK_ROOT."
        )
    MASK_ROOT = valid[0]

print("\nLOCKED DATA")
print("CACHE_DIR    :", CACHE_DIR)
print("TRAIN_RECORDS:", TRAIN_RECORDS, "| Fabric n=", len(train_rows))
print("VAL_RECORDS  :", VAL_RECORDS, "| Fabric n=", len(val_rows))
print("MASK_ROOT    :", MASK_ROOT)


In [ ]:

# ============================================================
# 6. CACHE PROVENANCE / BACKBONE HARD GATE
# ============================================================
manifest = json.loads(
    (CACHE_DIR / "manifest.json").read_text(encoding="utf-8")
)
signature = manifest.get("producer_signature", {})

print(
    "Day-05 locked backbone:",
    DAY05["locked"]["backbone"]["name"],
)
print("Cache producer signature:")
print(json.dumps(signature, indent=2, ensure_ascii=False))

sig_text = json.dumps(
    signature,
    ensure_ascii=False,
).lower()

if "vits16" in sig_text:
    raise RuntimeError(
        "BACKBONE DRIFT: cache có dấu hiệu ViT-S/16, "
        "trong khi Day-05/default khóa dinov3_vitb16."
    )

if "vitb16" not in sig_text:
    raise RuntimeError(
        "Không xác minh được cache là DINOv3 ViT-B/16. "
        "Không được full train với cache chưa xác minh."
    )

print("Cache backbone provenance: PASS")

old_ckpt = str(
    DAY04_PROTOCOL
    .get("frozen_backbone", {})
    .get("checkpoint", "")
)

if "vits16" in old_ckpt.lower():
    print(
        "\n[WARNING] day04_train_protocol.yaml hiện có checkpoint "
        "name chứa 'vits16', còn Day-05 khóa 'vitb16'.\n"
        "TV1-C runner KHÔNG dùng checkpoint field này; "
        "nó dùng cache provenance.\n"
        "Winner r,d cuối cùng cũng phải được chọn dưới "
        "CÙNG backbone ViT-B/16."
    )


In [ ]:

# ============================================================
# 7. FINAL r,d GATE — KHÔNG FABRICATE
# ============================================================
if ADAPTER_R is None or ADAPTER_D is None:
    raise RuntimeError(
        "STOP CÓ CHỦ ĐÍCH: chưa có Adapter winner Day-04.\n"
        "Khi selection xong, đặt:\n"
        "  ADAPTER_R = <winner_r>\n"
        "  ADAPTER_D = <winner_d>\n"
        "rồi chạy lại từ Cell 4.\n"
        "Không dùng r,d của smoke integration."
    )

ADAPTER_R = int(ADAPTER_R)
ADAPTER_D = int(ADAPTER_D)

if ADAPTER_R <= 0 or ADAPTER_D <= 0:
    raise ValueError("ADAPTER_R/D phải > 0.")

if ADAPTER_SELECTION_REPORT is not None:
    selection_path = Path(ADAPTER_SELECTION_REPORT)
    if not selection_path.is_file():
        raise FileNotFoundError(selection_path)
    print("Selection report:", selection_path)
else:
    print(
        "[WARNING] ADAPTER_SELECTION_REPORT=None. "
        "Nên lưu selection report để TV2 tái tạo chính xác."
    )

print(
    "FINAL Adapter lock:",
    {"r": ADAPTER_R, "d": ADAPTER_D},
)


In [ ]:

# ============================================================
# 8. DRY-RUN COMMANDS
# ============================================================
base = [
    sys.executable,
    "src/train/screen_representation.py",
    "--category", CATEGORY,
    "--r", str(ADAPTER_R),
    "--d", str(ADAPTER_D),
    "--cache-dir", str(CACHE_DIR),
    "--train-records", str(TRAIN_RECORDS),
    "--val-records", str(VAL_RECORDS),
    "--output-root", str(OUTPUT_ROOT),
    "--day05-config", str(DAY05_PATH),
    "--training-protocol", str(DAY04_PROTOCOL_PATH),
    "--seed", str(SEED),
    "--device", DEVICE,
    "--preflight-steps", str(PRECHECK_STEPS),
]

if MASK_ROOT is not None:
    base += ["--mask-root", str(MASK_ROOT)]

if ADAPTER_SELECTION_REPORT is not None:
    base += [
        "--adapter-selection-report",
        str(ADAPTER_SELECTION_REPORT),
    ]

if RESUME:
    base += ["--resume"]

for candidate in CANDIDATES:
    cmd = base + ["--candidate", candidate]
    print(
        " ".join(
            shlex.quote(str(x))
            for x in cmd
        )
    )


In [ ]:

# ============================================================
# 9. FULL TRAIN — R0 -> R1 -> R2
# ============================================================
# Đây là cell tốn thời gian thật.
# Mỗi candidate:
#   exact-final-(r,d) preflight
#   -> discard preflight model
#   -> fresh seeded model
#   -> full train
#   -> checkpoint every epoch
#
# Nếu Colab disconnect: chạy lại notebook với RESUME=True.

for candidate in CANDIDATES:
    print("\n" + "=" * 90)
    print("FULL TRAIN:", candidate, "×", CATEGORY)
    print("=" * 90)

    cmd = base + ["--candidate", candidate]

    r = subprocess.run(
        cmd,
        cwd=PROJECT_ROOT,
        text=True,
    )

    if r.returncode != 0:
        raise RuntimeError(
            f"{candidate} FAIL. "
            "Dừng ngay; không chạy candidate sau."
        )

print("\nAll 3 candidate commands completed.")


In [ ]:
# ============================================================
# 10. TV1-D — CHECKPOINT SELECTION + AUDIT
#      + selection_record.json
#
# ĐẶT NGAY SAU CELL 9 FULL TRAIN,
# TRƯỚC CELL "COMPLETENESS / REPRODUCIBILITY AUDIT"
# ============================================================
import csv
import math


def _read_csv_rows(
    path: Path,
) -> list[dict]:

    with Path(path).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:
        return list(
            csv.DictReader(f)
        )


def _file_sha256(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                8 * 1024 * 1024
            ),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def _select_with_day04_rule(
    rows: list[dict],
    monitor: str,
    mode: str,
):
    """
    Replay ĐÚNG logic Day-04:

        if is_better(
            score,
            best,
            mode
        ):
            replace best

    Day-04 is_better dùng strict:
        min -> value < best
        max -> value > best

    Vì vậy nếu bằng nhau:
        checkpoint cũ được giữ

    => tie_policy = earliest epoch.
    """

    best_score = None
    selected = None

    # Epoch tăng dần.
    for row in sorted(
        rows,
        key=lambda x: int(
            x["epoch"]
        ),
    ):

        if monitor not in row:
            raise RuntimeError(
                f"epoch_log thiếu "
                f"monitor={monitor!r}. "
                f"Columns="
                f"{sorted(row.keys())}"
            )

        score = float(
            row[monitor]
        )

        if not math.isfinite(score):
            raise RuntimeError(
                "Non-finite checkpoint "
                "metric: "
                f"epoch={row['epoch']}, "
                f"{monitor}={score}"
            )

        # CHÍNH comparator từ Day-04.
        if day04_is_better(
            score,
            best_score,
            mode,
        ):
            best_score = score
            selected = row

    if selected is None:
        raise RuntimeError(
            "Checkpoint selector "
            "không chọn được epoch nào."
        )

    return (
        selected,
        float(best_score),
    )


# ------------------------------------------------------------
# 1. RULE PHẢI ĐƯỢC LOCK TRƯỚC TRAIN
# ------------------------------------------------------------
if not TV1D_RULE_LOCK_PATH.is_file():
    raise RuntimeError(
        "Thiếu checkpoint_rule_lock.json. "
        "Bạn phải chạy Cell 4A "
        "TRƯỚC Full Train."
    )


locked_rule = json.loads(
    TV1D_RULE_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    locked_rule["rule_lock_sha256"]
    != TV1D_RULE_SHA256
):
    raise RuntimeError(
        "TV1-D rule lock SHA drift."
    )


TV1D_SELECTION_SUMMARY = {
    "schema":
        "msila.day05."
        "checkpoint_selection_summary.v1",

    "status": "RUNNING",

    "category": CATEGORY,
    "seed": int(SEED),

    "dev_split": "dev_synthetic",

    "rule_lock":
        str(TV1D_RULE_LOCK_PATH),

    "rule_lock_sha256":
        TV1D_RULE_SHA256,

    "day04_monitor":
        TV1D_DAY04_MONITOR,

    "day05_log_monitor":
        TV1D_LOG_MONITOR,

    "mode":
        TV1D_MODE,

    "selector":
        "src.train.screen_adapter."
        "is_better",

    "tie_policy":
        "keep_earliest_epoch",

    "runs": {},
}


# ============================================================
# 2. SELECT + VERIFY R0 / R1 / R2
# ============================================================
for candidate in CANDIDATES:

    run_dir = (
        OUTPUT_ROOT
        / f"seed_{SEED}"
        / CATEGORY
        / candidate
    )

    epoch_log_path = (
        run_dir
        / "epoch_log.csv"
    )

    best_path = (
        run_dir
        / "best.pt"
    )

    resolved_path = (
        run_dir
        / "resolved_config.yaml"
    )

    manifest_path = (
        run_dir
        / "run_manifest.json"
    )

    selection_path = (
        run_dir
        / "selection_record.json"
    )


    # --------------------------------------------------------
    # 2.1 REQUIRED ARTIFACTS
    # --------------------------------------------------------
    for required in (
        epoch_log_path,
        best_path,
        resolved_path,
        manifest_path,
    ):

        if not required.is_file():
            raise RuntimeError(
                f"{candidate}: missing "
                f"required TV1-D artifact: "
                f"{required}"
            )


    # --------------------------------------------------------
    # 2.2 FULL TRAIN PHẢI COMPLETE
    # --------------------------------------------------------
    manifest = json.loads(
        manifest_path.read_text(
            encoding="utf-8"
        )
    )

    if (
        manifest.get("status")
        != "COMPLETE"
    ):
        raise RuntimeError(
            f"{candidate}: full train "
            "chưa COMPLETE, "
            "không được checkpoint selection."
        )


    # --------------------------------------------------------
    # 2.3 RULE PHẢI LOCK TRƯỚC RUN
    # --------------------------------------------------------
    if (
        float(
            locked_rule[
                "locked_at_unix"
            ]
        )
        >
        float(
            manifest[
                "started_at_unix"
            ]
        )
    ):
        raise RuntimeError(
            f"{candidate}: "
            "checkpoint rule được lock "
            "SAU khi run bắt đầu."
        )


    # --------------------------------------------------------
    # 2.4 RESOLVED CONFIG PHẢI DÙNG CÙNG RULE
    # --------------------------------------------------------
    resolved = yaml.safe_load(
        resolved_path.read_text(
            encoding="utf-8"
        )
    )

    runner_rule = resolved.get(
        "checkpoint_rule",
        {},
    )

    if (
        str(
            runner_rule.get(
                "monitor"
            )
        )
        != TV1D_LOG_MONITOR
        or
        str(
            runner_rule.get(
                "mode"
            )
        ).lower()
        != TV1D_MODE
    ):
        raise RuntimeError(
            f"{candidate}: "
            "runner checkpoint rule drift.\n"
            f"resolved_config="
            f"{runner_rule}\n"
            f"expected monitor="
            f"{TV1D_LOG_MONITOR}, "
            f"mode={TV1D_MODE}"
        )


    # --------------------------------------------------------
    # 2.5 LOAD FULL EPOCH LOG
    # --------------------------------------------------------
    rows = _read_csv_rows(
        epoch_log_path
    )

    if not rows:
        raise RuntimeError(
            f"{candidate}: "
            "epoch_log.csv rỗng."
        )


    epochs = [
        int(row["epoch"])
        for row in rows
    ]

    expected_epochs = int(
        DAY04_PROTOCOL[
            "training"
        ][
            "epochs"
        ]
    )


    # Không duplicate epoch.
    if len(epochs) != len(
        set(epochs)
    ):
        raise RuntimeError(
            f"{candidate}: "
            "epoch_log có duplicate epoch: "
            f"{epochs}"
        )


    # Phải đủ toàn bộ 1..epochs.
    if epochs != list(
        range(
            1,
            expected_epochs + 1,
        )
    ):
        raise RuntimeError(
            f"{candidate}: epoch_log "
            f"không đủ liên tục "
            f"1..{expected_epochs}. "
            f"Observed count="
            f"{len(epochs)}, "
            f"first={epochs[:3]}, "
            f"last={epochs[-3:]}"
        )


    # --------------------------------------------------------
    # 2.6 REPLAY CHÍNH CHECKPOINT SELECTOR DAY-04
    # --------------------------------------------------------
    (
        selected_row,
        selected_metric,
    ) = _select_with_day04_rule(
        rows,
        monitor=TV1D_LOG_MONITOR,
        mode=TV1D_MODE,
    )

    selected_epoch = int(
        selected_row["epoch"]
    )


    # --------------------------------------------------------
    # 2.7 best.pt PHẢI ĐÚNG EPOCH VỪA CHỌN
    # --------------------------------------------------------
    ckpt = torch.load(
        best_path,
        map_location="cpu",
        weights_only=False,
    )

    checkpoint_epoch = int(
        ckpt["epoch"]
    )

    checkpoint_best_val = float(
        ckpt["best_val"]
    )


    if (
        checkpoint_epoch
        != selected_epoch
    ):
        raise RuntimeError(
            f"{candidate}: "
            f"best.pt epoch="
            f"{checkpoint_epoch} "
            f"nhưng Day-04 rule "
            f"chọn epoch="
            f"{selected_epoch}."
        )


    if not math.isclose(
        checkpoint_best_val,
        selected_metric,
        rel_tol=0.0,
        abs_tol=1e-12,
    ):
        raise RuntimeError(
            f"{candidate}: "
            f"best.pt metric="
            f"{checkpoint_best_val} "
            f"!= selected metric="
            f"{selected_metric}."
        )


    # best.pt phải thuộc đúng resolved config của run.
    if (
        ckpt.get(
            "resolved_config_sha256"
        )
        != manifest.get(
            "resolved_config_sha256"
        )
    ):
        raise RuntimeError(
            f"{candidate}: "
            "best.pt không thuộc "
            "resolved_config của run."
        )


    # ========================================================
    # 2.8 WRITE selection_record.json
    # ========================================================
    selection_record = {

        "schema":
            "msila.day05."
            "checkpoint_selection_record.v1",

        "status":
            "PASS",

        "candidate":
            candidate,

        "category":
            CATEGORY,

        "seed":
            int(SEED),

        "dev_split":
            "dev_synthetic",


        # ----------------------------------------------------
        # RULE PROVENANCE
        # ----------------------------------------------------
        "rule": {

            "rule_source":
                str(
                    DAY04_PROTOCOL_PATH
                ),

            "rule_source_sha256":
                DAY04_PROTOCOL_SHA256,

            "rule_lock_path":
                str(
                    TV1D_RULE_LOCK_PATH
                ),

            "rule_lock_sha256":
                TV1D_RULE_SHA256,

            "selector_source":
                "src.train."
                "screen_adapter.is_better",

            "selector_file_sha256":
                DAY04_SELECTOR_SHA256,

            "day04_monitor":
                TV1D_DAY04_MONITOR,

            "day05_log_monitor":
                TV1D_LOG_MONITOR,

            "mode":
                TV1D_MODE,

            "comparison":
                "strict_improvement",

            "tie_policy":
                "keep_earliest_epoch",
        },


        # ----------------------------------------------------
        # SELECTED CHECKPOINT
        # ----------------------------------------------------
        "selection": {

            "selected_epoch":
                selected_epoch,

            "selected_metric":
                selected_metric,

            "metric_name":
                TV1D_LOG_MONITOR,

            "checkpoint":
                str(best_path),

            "checkpoint_sha256":
                _file_sha256(
                    best_path
                ),
        },


        # ----------------------------------------------------
        # AUDIT EVIDENCE
        # ----------------------------------------------------
        "evidence": {

            "epoch_log":
                str(
                    epoch_log_path
                ),

            "epoch_log_sha256":
                _file_sha256(
                    epoch_log_path
                ),

            "num_logged_epochs":
                len(rows),

            "resolved_config":
                str(
                    resolved_path
                ),

            "resolved_config_sha256_file":
                _file_sha256(
                    resolved_path
                ),

            "run_manifest":
                str(
                    manifest_path
                ),

            "git_commit":
                manifest[
                    "git_commit"
                ],

            "protocol_lock_sha256":
                manifest[
                    "protocol_lock_sha256"
                ],

            "checkpoint_resolved_config_sha256":
                ckpt[
                    "resolved_config_sha256"
                ],
        },
    }


    selection_path.write_text(
        json.dumps(
            selection_record,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    # ========================================================
    # 2.9 REGISTER TV1-D INTO run_manifest.json
    # ========================================================
    manifest.setdefault(
        "artifacts",
        {},
    )

    manifest[
        "artifacts"
    ][
        "selection_record"
    ] = str(
        selection_path
    )

    manifest[
        "checkpoint_selection"
    ] = {

        "status":
            "PASS",

        "selected_epoch":
            selected_epoch,

        "selected_metric":
            selected_metric,

        "monitor":
            TV1D_LOG_MONITOR,

        "mode":
            TV1D_MODE,

        "rule_lock_sha256":
            TV1D_RULE_SHA256,
    }


    manifest_path.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    # ========================================================
    # 2.10 ADD TO GLOBAL TV1-D SUMMARY
    # ========================================================
    TV1D_SELECTION_SUMMARY[
        "runs"
    ][
        candidate
    ] = {

        "status":
            "PASS",

        "selected_epoch":
            selected_epoch,

        "selected_metric":
            selected_metric,

        "checkpoint":
            str(best_path),

        "checkpoint_sha256":
            _file_sha256(
                best_path
            ),

        "selection_record":
            str(
                selection_path
            ),

        "selection_record_sha256":
            _file_sha256(
                selection_path
            ),

        "git_commit":
            manifest[
                "git_commit"
            ],

        "protocol_lock_sha256":
            manifest[
                "protocol_lock_sha256"
            ],
    }


# ============================================================
# 3. GLOBAL 3/3 GATES
# ============================================================

# Phải đủ chính xác R0,R1,R2.
if (
    set(
        TV1D_SELECTION_SUMMARY[
            "runs"
        ]
    )
    != set(CANDIDATES)
):
    raise RuntimeError(
        "TV1-D không đủ "
        "3/3 candidate."
    )


# Cùng commit.
if len({
    x["git_commit"]
    for x in
    TV1D_SELECTION_SUMMARY[
        "runs"
    ].values()
}) != 1:
    raise RuntimeError(
        "R0/R1/R2 không cùng "
        "git commit."
    )


# Cùng protocol full train.
if len({
    x["protocol_lock_sha256"]
    for x in
    TV1D_SELECTION_SUMMARY[
        "runs"
    ].values()
}) != 1:
    raise RuntimeError(
        "R0/R1/R2 không cùng "
        "protocol lock."
    )


TV1D_SELECTION_SUMMARY[
    "status"
] = "PASS"


# ============================================================
# 4. GLOBAL SUMMARY ARTIFACT
# ============================================================
TV1D_SUMMARY_PATH = (
    OUTPUT_ROOT
    / f"seed_{SEED}"
    / CATEGORY
    / "tv1d_checkpoint_selection_summary.json"
)

TV1D_SUMMARY_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

TV1D_SUMMARY_PATH.write_text(
    json.dumps(
        TV1D_SELECTION_SUMMARY,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 90)
print(
    "TV1-D CHECKPOINT SELECTION: PASS"
)
print("=" * 90)

print(
    "Rule:",
    f"{TV1D_DAY04_MONITOR} "
    f"-> {TV1D_LOG_MONITOR}",
    TV1D_MODE,
)

for candidate in CANDIDATES:

    x = TV1D_SELECTION_SUMMARY[
        "runs"
    ][
        candidate
    ]

    print(
        f"{candidate}: "
        f"selected epoch="
        f"{x['selected_epoch']} | "
        f"{TV1D_LOG_MONITOR}="
        f"{x['selected_metric']:.8f}"
    )

    print(
        "   best.pt          :",
        x["checkpoint"],
    )

    print(
        "   selection_record :",
        x["selection_record"],
    )


print(
    "3/3 selection summary:",
    TV1D_SUMMARY_PATH,
)

In [ ]:
# ============================================================
# 11. TV1-C + TV1-D
#     COMPLETENESS / REPRODUCIBILITY AUDIT
#
# THAY TOÀN BỘ CELL 10 CŨ BẰNG CELL NÀY
# ============================================================

required_files = [
    "training_log.csv",
    "epoch_log.csv",

    "best.pt",
    "last.pt",

    "resolved_config.yaml",
    "run_manifest.json",

    "preflight_report.json",

    "sample_anomaly_map.png",

    # TV1-D bắt buộc.
    "selection_record.json",
]


summary = {}


for candidate in CANDIDATES:

    run_dir = (
        OUTPUT_ROOT
        / f"seed_{SEED}"
        / CATEGORY
        / candidate
    )


    # --------------------------------------------------------
    # 1. ARTIFACT COMPLETENESS
    # --------------------------------------------------------
    missing = [
        name
        for name in required_files
        if not (
            run_dir
            / name
        ).is_file()
    ]


    if missing:
        raise RuntimeError(
            f"{candidate}: "
            f"missing artifacts "
            f"{missing}"
        )


    # --------------------------------------------------------
    # 2. RUN MANIFEST
    # --------------------------------------------------------
    manifest = json.loads(
        (
            run_dir
            / "run_manifest.json"
        ).read_text(
            encoding="utf-8"
        )
    )


    if (
        manifest.get("status")
        != "COMPLETE"
    ):
        raise RuntimeError(
            f"{candidate}: "
            "manifest status "
            "!= COMPLETE"
        )


    # --------------------------------------------------------
    # 3. TV1-D MANIFEST GATE
    # --------------------------------------------------------
    ckpt_sel = manifest.get(
        "checkpoint_selection",
        {},
    )


    if (
        ckpt_sel.get("status")
        != "PASS"
    ):
        raise RuntimeError(
            f"{candidate}: "
            "TV1-D checkpoint_selection "
            "!= PASS"
        )


    if (
        ckpt_sel.get(
            "rule_lock_sha256"
        )
        != TV1D_RULE_SHA256
    ):
        raise RuntimeError(
            f"{candidate}: "
            "TV1-D rule lock drift"
        )


    # --------------------------------------------------------
    # 4. selection_record.json
    # --------------------------------------------------------
    selection_record = json.loads(
        (
            run_dir
            / "selection_record.json"
        ).read_text(
            encoding="utf-8"
        )
    )


    if (
        selection_record.get(
            "status"
        )
        != "PASS"
    ):
        raise RuntimeError(
            f"{candidate}: "
            "selection_record "
            "status != PASS"
        )


    if (
        int(
            selection_record[
                "selection"
            ][
                "selected_epoch"
            ]
        )
        !=
        int(
            ckpt_sel[
                "selected_epoch"
            ]
        )
    ):
        raise RuntimeError(
            f"{candidate}: "
            "selected epoch mismatch "
            "between manifest "
            "and selection_record"
        )


    # --------------------------------------------------------
    # 5. SUMMARY
    # --------------------------------------------------------
    summary[candidate] = {

        "run_dir":
            str(run_dir),

        "completed_epochs":
            manifest[
                "completed_epochs"
            ],

        "global_step":
            manifest[
                "global_step"
            ],

        "best_val_total_loss":
            manifest[
                "best_val_total_loss"
            ],

        "git_commit":
            manifest[
                "git_commit"
            ],

        "protocol_lock_sha256":
            manifest[
                "protocol_lock_sha256"
            ],

        "selected_epoch":
            selection_record[
                "selection"
            ][
                "selected_epoch"
            ],

        "selected_metric":
            selection_record[
                "selection"
            ][
                "selected_metric"
            ],

        "selection_record":
            str(
                run_dir
                / "selection_record.json"
            ),
    }


# ============================================================
# 6. FAIRNESS GATES ACROSS R0/R1/R2
# ============================================================

# Cùng code.
assert len({
    x["git_commit"]
    for x in summary.values()
}) == 1


# Cùng full-training protocol.
assert len({
    x["protocol_lock_sha256"]
    for x in summary.values()
}) == 1


# ============================================================
# 7. GLOBAL TV1-D SUMMARY PHẢI PASS
# ============================================================
tv1d_summary_path = (
    OUTPUT_ROOT
    / f"seed_{SEED}"
    / CATEGORY
    / "tv1d_checkpoint_selection_summary.json"
)


if not tv1d_summary_path.is_file():

    raise RuntimeError(
        "Missing TV1-D summary: "
        f"{tv1d_summary_path}"
    )


tv1d_summary = json.loads(
    tv1d_summary_path.read_text(
        encoding="utf-8"
    )
)


if (
    tv1d_summary.get("status")
    != "PASS"
):
    raise RuntimeError(
        "TV1-D checkpoint selection "
        "summary != PASS"
    )


if (
    set(
        tv1d_summary.get(
            "runs",
            {},
        )
    )
    != set(CANDIDATES)
):
    raise RuntimeError(
        "TV1-D summary "
        "không đủ R0/R1/R2."
    )


if (
    tv1d_summary.get(
        "rule_lock_sha256"
    )
    != TV1D_RULE_SHA256
):
    raise RuntimeError(
        "TV1-D summary "
        "checkpoint rule drift."
    )


# ============================================================
# 8. FINAL TV1-C + TV1-D COMPLETENESS
# ============================================================
complete_path = (
    OUTPUT_ROOT
    / f"seed_{SEED}"
    / CATEGORY
    / "tv1c_tv1d_completeness.json"
)


complete_path.write_text(
    json.dumps(
        {
            "status":
                "PASS",

            "tv1c_full_train":
                "PASS",

            "tv1d_checkpoint_selection":
                "PASS",

            "category":
                CATEGORY,

            "seed":
                SEED,

            "adapter": {
                "r":
                    ADAPTER_R,

                "d":
                    ADAPTER_D,
            },

            "checkpoint_rule": {

                "day04_monitor":
                    TV1D_DAY04_MONITOR,

                "day05_log_monitor":
                    TV1D_LOG_MONITOR,

                "mode":
                    TV1D_MODE,

                "selector":
                    "src.train."
                    "screen_adapter."
                    "is_better",

                "rule_lock_sha256":
                    TV1D_RULE_SHA256,
            },

            "runs":
                summary,

            "tv1d_summary":
                str(
                    tv1d_summary_path
                ),
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 90)

print(
    "TV1-C + TV1-D "
    "COMPLETENESS: PASS"
)

print("=" * 90)


for c, x in summary.items():

    print(
        c,
        "->",
        x["run_dir"],
    )

    print(
        "   epochs:",
        x["completed_epochs"],
        "steps:",
        x["global_step"],
    )

    print(
        "   selected epoch:",
        x["selected_epoch"],
        "| selected metric:",
        x["selected_metric"],
    )

    print(
        "   selection record:",
        x["selection_record"],
    )


print(
    "TV1-D summary :",
    tv1d_summary_path,
)

print(
    "Completeness  :",
    complete_path,
)

In [ ]:
# ============================================================
# 10. TV1-D — CHECKPOINT SELECTION + AUDIT
#      + selection_record.json
#
# ĐẶT NGAY SAU CELL 9 FULL TRAIN,
# TRƯỚC CELL "COMPLETENESS / REPRODUCIBILITY AUDIT"
# ============================================================
import csv
import math


def _read_csv_rows(
    path: Path,
) -> list[dict]:

    with Path(path).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:
        return list(
            csv.DictReader(f)
        )


def _file_sha256(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                8 * 1024 * 1024
            ),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def _select_with_day04_rule(
    rows: list[dict],
    monitor: str,
    mode: str,
):
    """
    Replay ĐÚNG logic Day-04:

        if is_better(
            score,
            best,
            mode
        ):
            replace best

    Day-04 is_better dùng strict:
        min -> value < best
        max -> value > best

    Vì vậy nếu bằng nhau:
        checkpoint cũ được giữ

    => tie_policy = earliest epoch.
    """

    best_score = None
    selected = None

    # Epoch tăng dần.
    for row in sorted(
        rows,
        key=lambda x: int(
            x["epoch"]
        ),
    ):

        if monitor not in row:
            raise RuntimeError(
                f"epoch_log thiếu "
                f"monitor={monitor!r}. "
                f"Columns="
                f"{sorted(row.keys())}"
            )

        score = float(
            row[monitor]
        )

        if not math.isfinite(score):
            raise RuntimeError(
                "Non-finite checkpoint "
                "metric: "
                f"epoch={row['epoch']}, "
                f"{monitor}={score}"
            )

        # CHÍNH comparator từ Day-04.
        if day04_is_better(
            score,
            best_score,
            mode,
        ):
            best_score = score
            selected = row

    if selected is None:
        raise RuntimeError(
            "Checkpoint selector "
            "không chọn được epoch nào."
        )

    return (
        selected,
        float(best_score),
    )


# ------------------------------------------------------------
# 1. RULE PHẢI ĐƯỢC LOCK TRƯỚC TRAIN
# ------------------------------------------------------------
if not TV1D_RULE_LOCK_PATH.is_file():
    raise RuntimeError(
        "Thiếu checkpoint_rule_lock.json. "
        "Bạn phải chạy Cell 4A "
        "TRƯỚC Full Train."
    )


locked_rule = json.loads(
    TV1D_RULE_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    locked_rule["rule_lock_sha256"]
    != TV1D_RULE_SHA256
):
    raise RuntimeError(
        "TV1-D rule lock SHA drift."
    )


TV1D_SELECTION_SUMMARY = {
    "schema":
        "msila.day05."
        "checkpoint_selection_summary.v1",

    "status": "RUNNING",

    "category": CATEGORY,
    "seed": int(SEED),

    "dev_split": "dev_synthetic",

    "rule_lock":
        str(TV1D_RULE_LOCK_PATH),

    "rule_lock_sha256":
        TV1D_RULE_SHA256,

    "day04_monitor":
        TV1D_DAY04_MONITOR,

    "day05_log_monitor":
        TV1D_LOG_MONITOR,

    "mode":
        TV1D_MODE,

    "selector":
        "src.train.screen_adapter."
        "is_better",

    "tie_policy":
        "keep_earliest_epoch",

    "runs": {},
}


# ============================================================
# 2. SELECT + VERIFY R0 / R1 / R2
# ============================================================
for candidate in CANDIDATES:

    run_dir = (
        OUTPUT_ROOT
        / f"seed_{SEED}"
        / CATEGORY
        / candidate
    )

    epoch_log_path = (
        run_dir
        / "epoch_log.csv"
    )

    best_path = (
        run_dir
        / "best.pt"
    )

    resolved_path = (
        run_dir
        / "resolved_config.yaml"
    )

    manifest_path = (
        run_dir
        / "run_manifest.json"
    )

    selection_path = (
        run_dir
        / "selection_record.json"
    )


    # --------------------------------------------------------
    # 2.1 REQUIRED ARTIFACTS
    # --------------------------------------------------------
    for required in (
        epoch_log_path,
        best_path,
        resolved_path,
        manifest_path,
    ):

        if not required.is_file():
            raise RuntimeError(
                f"{candidate}: missing "
                f"required TV1-D artifact: "
                f"{required}"
            )


    # --------------------------------------------------------
    # 2.2 FULL TRAIN PHẢI COMPLETE
    # --------------------------------------------------------
    manifest = json.loads(
        manifest_path.read_text(
            encoding="utf-8"
        )
    )

    if (
        manifest.get("status")
        != "COMPLETE"
    ):
        raise RuntimeError(
            f"{candidate}: full train "
            "chưa COMPLETE, "
            "không được checkpoint selection."
        )


    # --------------------------------------------------------
    # 2.3 RULE PHẢI LOCK TRƯỚC RUN
    # --------------------------------------------------------
    if (
        float(
            locked_rule[
                "locked_at_unix"
            ]
        )
        >
        float(
            manifest[
                "started_at_unix"
            ]
        )
    ):
        raise RuntimeError(
            f"{candidate}: "
            "checkpoint rule được lock "
            "SAU khi run bắt đầu."
        )


    # --------------------------------------------------------
    # 2.4 RESOLVED CONFIG PHẢI DÙNG CÙNG RULE
    # --------------------------------------------------------
    resolved = yaml.safe_load(
        resolved_path.read_text(
            encoding="utf-8"
        )
    )

    runner_rule = resolved.get(
        "checkpoint_rule",
        {},
    )

    if (
        str(
            runner_rule.get(
                "monitor"
            )
        )
        != TV1D_LOG_MONITOR
        or
        str(
            runner_rule.get(
                "mode"
            )
        ).lower()
        != TV1D_MODE
    ):
        raise RuntimeError(
            f"{candidate}: "
            "runner checkpoint rule drift.\n"
            f"resolved_config="
            f"{runner_rule}\n"
            f"expected monitor="
            f"{TV1D_LOG_MONITOR}, "
            f"mode={TV1D_MODE}"
        )


    # --------------------------------------------------------
    # 2.5 LOAD FULL EPOCH LOG
    # --------------------------------------------------------
    rows = _read_csv_rows(
        epoch_log_path
    )

    if not rows:
        raise RuntimeError(
            f"{candidate}: "
            "epoch_log.csv rỗng."
        )


    epochs = [
        int(row["epoch"])
        for row in rows
    ]

    expected_epochs = int(
        DAY04_PROTOCOL[
            "training"
        ][
            "epochs"
        ]
    )


    # Không duplicate epoch.
    if len(epochs) != len(
        set(epochs)
    ):
        raise RuntimeError(
            f"{candidate}: "
            "epoch_log có duplicate epoch: "
            f"{epochs}"
        )


    # Phải đủ toàn bộ 1..epochs.
    if epochs != list(
        range(
            1,
            expected_epochs + 1,
        )
    ):
        raise RuntimeError(
            f"{candidate}: epoch_log "
            f"không đủ liên tục "
            f"1..{expected_epochs}. "
            f"Observed count="
            f"{len(epochs)}, "
            f"first={epochs[:3]}, "
            f"last={epochs[-3:]}"
        )


    # --------------------------------------------------------
    # 2.6 REPLAY CHÍNH CHECKPOINT SELECTOR DAY-04
    # --------------------------------------------------------
    (
        selected_row,
        selected_metric,
    ) = _select_with_day04_rule(
        rows,
        monitor=TV1D_LOG_MONITOR,
        mode=TV1D_MODE,
    )

    selected_epoch = int(
        selected_row["epoch"]
    )


    # --------------------------------------------------------
    # 2.7 best.pt PHẢI ĐÚNG EPOCH VỪA CHỌN
    # --------------------------------------------------------
    ckpt = torch.load(
        best_path,
        map_location="cpu",
        weights_only=False,
    )

    checkpoint_epoch = int(
        ckpt["epoch"]
    )

    checkpoint_best_val = float(
        ckpt["best_val"]
    )


    if (
        checkpoint_epoch
        != selected_epoch
    ):
        raise RuntimeError(
            f"{candidate}: "
            f"best.pt epoch="
            f"{checkpoint_epoch} "
            f"nhưng Day-04 rule "
            f"chọn epoch="
            f"{selected_epoch}."
        )


    if not math.isclose(
        checkpoint_best_val,
        selected_metric,
        rel_tol=0.0,
        abs_tol=1e-12,
    ):
        raise RuntimeError(
            f"{candidate}: "
            f"best.pt metric="
            f"{checkpoint_best_val} "
            f"!= selected metric="
            f"{selected_metric}."
        )


    # best.pt phải thuộc đúng resolved config của run.
    if (
        ckpt.get(
            "resolved_config_sha256"
        )
        != manifest.get(
            "resolved_config_sha256"
        )
    ):
        raise RuntimeError(
            f"{candidate}: "
            "best.pt không thuộc "
            "resolved_config của run."
        )


    # ========================================================
    # 2.8 WRITE selection_record.json
    # ========================================================
    selection_record = {

        "schema":
            "msila.day05."
            "checkpoint_selection_record.v1",

        "status":
            "PASS",

        "candidate":
            candidate,

        "category":
            CATEGORY,

        "seed":
            int(SEED),

        "dev_split":
            "dev_synthetic",


        # ----------------------------------------------------
        # RULE PROVENANCE
        # ----------------------------------------------------
        "rule": {

            "rule_source":
                str(
                    DAY04_PROTOCOL_PATH
                ),

            "rule_source_sha256":
                DAY04_PROTOCOL_SHA256,

            "rule_lock_path":
                str(
                    TV1D_RULE_LOCK_PATH
                ),

            "rule_lock_sha256":
                TV1D_RULE_SHA256,

            "selector_source":
                "src.train."
                "screen_adapter.is_better",

            "selector_file_sha256":
                DAY04_SELECTOR_SHA256,

            "day04_monitor":
                TV1D_DAY04_MONITOR,

            "day05_log_monitor":
                TV1D_LOG_MONITOR,

            "mode":
                TV1D_MODE,

            "comparison":
                "strict_improvement",

            "tie_policy":
                "keep_earliest_epoch",
        },


        # ----------------------------------------------------
        # SELECTED CHECKPOINT
        # ----------------------------------------------------
        "selection": {

            "selected_epoch":
                selected_epoch,

            "selected_metric":
                selected_metric,

            "metric_name":
                TV1D_LOG_MONITOR,

            "checkpoint":
                str(best_path),

            "checkpoint_sha256":
                _file_sha256(
                    best_path
                ),
        },


        # ----------------------------------------------------
        # AUDIT EVIDENCE
        # ----------------------------------------------------
        "evidence": {

            "epoch_log":
                str(
                    epoch_log_path
                ),

            "epoch_log_sha256":
                _file_sha256(
                    epoch_log_path
                ),

            "num_logged_epochs":
                len(rows),

            "resolved_config":
                str(
                    resolved_path
                ),

            "resolved_config_sha256_file":
                _file_sha256(
                    resolved_path
                ),

            "run_manifest":
                str(
                    manifest_path
                ),

            "git_commit":
                manifest[
                    "git_commit"
                ],

            "protocol_lock_sha256":
                manifest[
                    "protocol_lock_sha256"
                ],

            "checkpoint_resolved_config_sha256":
                ckpt[
                    "resolved_config_sha256"
                ],
        },
    }


    selection_path.write_text(
        json.dumps(
            selection_record,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    # ========================================================
    # 2.9 REGISTER TV1-D INTO run_manifest.json
    # ========================================================
    manifest.setdefault(
        "artifacts",
        {},
    )

    manifest[
        "artifacts"
    ][
        "selection_record"
    ] = str(
        selection_path
    )

    manifest[
        "checkpoint_selection"
    ] = {

        "status":
            "PASS",

        "selected_epoch":
            selected_epoch,

        "selected_metric":
            selected_metric,

        "monitor":
            TV1D_LOG_MONITOR,

        "mode":
            TV1D_MODE,

        "rule_lock_sha256":
            TV1D_RULE_SHA256,
    }


    manifest_path.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    # ========================================================
    # 2.10 ADD TO GLOBAL TV1-D SUMMARY
    # ========================================================
    TV1D_SELECTION_SUMMARY[
        "runs"
    ][
        candidate
    ] = {

        "status":
            "PASS",

        "selected_epoch":
            selected_epoch,

        "selected_metric":
            selected_metric,

        "checkpoint":
            str(best_path),

        "checkpoint_sha256":
            _file_sha256(
                best_path
            ),

        "selection_record":
            str(
                selection_path
            ),

        "selection_record_sha256":
            _file_sha256(
                selection_path
            ),

        "git_commit":
            manifest[
                "git_commit"
            ],

        "protocol_lock_sha256":
            manifest[
                "protocol_lock_sha256"
            ],
    }


# ============================================================
# 3. GLOBAL 3/3 GATES
# ============================================================

# Phải đủ chính xác R0,R1,R2.
if (
    set(
        TV1D_SELECTION_SUMMARY[
            "runs"
        ]
    )
    != set(CANDIDATES)
):
    raise RuntimeError(
        "TV1-D không đủ "
        "3/3 candidate."
    )


# Cùng commit.
if len({
    x["git_commit"]
    for x in
    TV1D_SELECTION_SUMMARY[
        "runs"
    ].values()
}) != 1:
    raise RuntimeError(
        "R0/R1/R2 không cùng "
        "git commit."
    )


# Cùng protocol full train.
if len({
    x["protocol_lock_sha256"]
    for x in
    TV1D_SELECTION_SUMMARY[
        "runs"
    ].values()
}) != 1:
    raise RuntimeError(
        "R0/R1/R2 không cùng "
        "protocol lock."
    )


TV1D_SELECTION_SUMMARY[
    "status"
] = "PASS"


# ============================================================
# 4. GLOBAL SUMMARY ARTIFACT
# ============================================================
TV1D_SUMMARY_PATH = (
    OUTPUT_ROOT
    / f"seed_{SEED}"
    / CATEGORY
    / "tv1d_checkpoint_selection_summary.json"
)

TV1D_SUMMARY_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

TV1D_SUMMARY_PATH.write_text(
    json.dumps(
        TV1D_SELECTION_SUMMARY,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 90)
print(
    "TV1-D CHECKPOINT SELECTION: PASS"
)
print("=" * 90)

print(
    "Rule:",
    f"{TV1D_DAY04_MONITOR} "
    f"-> {TV1D_LOG_MONITOR}",
    TV1D_MODE,
)

for candidate in CANDIDATES:

    x = TV1D_SELECTION_SUMMARY[
        "runs"
    ][
        candidate
    ]

    print(
        f"{candidate}: "
        f"selected epoch="
        f"{x['selected_epoch']} | "
        f"{TV1D_LOG_MONITOR}="
        f"{x['selected_metric']:.8f}"
    )

    print(
        "   best.pt          :",
        x["checkpoint"],
    )

    print(
        "   selection_record :",
        x["selection_record"],
    )


print(
    "3/3 selection summary:",
    TV1D_SUMMARY_PATH,
)


## Output

```text
outputs/day05/full_train/
└── seed_42/
    └── fabric/
        ├── R0/
        │   ├── resolved_config.yaml
        │   ├── run_manifest.json
        │   ├── preflight_report.json
        │   ├── training_log.csv
        │   ├── epoch_log.csv
        │   ├── best.pt
        │   ├── last.pt
        │   ├── sample_anomaly_map.png
        │   └── sample_anomaly_map.npy
        ├── R1/
        └── R2/
```

`training_log.csv` có từng optimizer step:

`epoch, step_in_epoch, global_step, total_loss, bce, dice, lr, grad_norm, elapsed_sec, peak_vram_mb`.

`run_manifest.json` lưu candidate/category/seed, `(r,d)`, Git branch+commit, dataset fingerprints, cache producer hash, checkpoint rule và trạng thái run.

TV1-C tạo **checkpoint + bằng chứng training thật**. Việc kết luận R0/R1/R2 cái nào tốt hơn phải nằm ở bước evaluator đã khóa, không dựa chỉ vào training loss.
